# Dinov2-BigEarthS2 — Training Pipeline

Multi-label classification of **BigEarthNet-S2** (Sentinel-2, 590,326 patches, 43 classes) with a **DINOv2-B** backbone (86M params) and a custom 2-layer head.

**Kaggle setup:**
- Accelerator: **GPU P100** (preferred) or **T4 ×2**
- Internet: **ON** (package install + HF streaming + timm weights)
- Persistence: **Files** (so checkpoints survive)

**Pipeline:** RGB-only bands (B04/B03/B02) → 3-phase fine-tuning (head → last 2 blocks → full) → checkpoint every epoch + curves every 5 epochs + history JSON.

> The architecture, class list, and checkpoint format here **exactly match** the FastAPI backend in `backend/app/models/dinov2.py` and `backend/app/config.py`. The trained `model_best.pth` drops straight into `backend/checkpoints/`.

## 1 · Install dependencies

In [ ]:
# Core ML stack is preinstalled on Kaggle; we add the extras.
!pip install -q timm datasets scikit-learn matplotlib seaborn tensorboard

In [ ]:
import os, sys, json, math, random, time, copy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.utils.tensorboard import SummaryWriter

import torchvision.transforms as T
from PIL import Image

from sklearn.metrics import f1_score, hamming_loss
import matplotlib.pyplot as plt

import timm

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Device + multi-GPU detection
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count()
print(f'Python  : {sys.version.split()[0]}')
print(f'Torch   : {torch.__version__}')
print(f'timm    : {timm.__version__}')
print(f'Device  : {DEVICE}  |  GPUs available: {N_GPUS}')

## 2 · Configuration

All hyperparameters in one place. `CLASSES_43` must match `backend/app/config.py` exactly.

In [ ]:
CFG = {
    # --- Model ---
    'backbone': 'vit_base_patch14_dinov2.lvd142m',  # DINOv2-B, 86M params, 768-d features
    'backbone_dim': 768,
    'num_classes': 43,
    'head_dropout': 0.3,
    'image_size': 224,

    # --- Data ---
    'batch_size': 128 if N_GPUS >= 2 else 64,   # 128 for T4x2 (DataParallel), 64 for P100
    'num_workers': 4,
    'grad_accum_steps': 2,                       # simulate 2x effective batch

    # --- Three-phase fine-tuning ---
    'phases': [
        {'name': 'head',         'epochs': 5,  'lr': 1e-3, 'unfreeze': 'head',          'warmup': False},
        {'name': 'last_2_blocks','epochs': 10, 'lr': 1e-4, 'unfreeze': 'last_2_blocks', 'warmup': False},
        {'name': 'full',         'epochs': 15, 'lr': 1e-5, 'unfreeze': 'full',          'warmup': False},
    ],

    # --- Optimizer / scheduler ---
    'optimizer': 'adamw',
    'weight_decay': 0.01,
    'betas': (0.9, 0.999),
    'sched_mode': 'max',
    'sched_factor': 0.5,
    'sched_patience': 3,
    'sched_min_lr': 1e-7,

    # --- Early stopping ---
    'es_patience': 7,
    'es_min_delta': 0.001,

    # --- Inference / metrics ---
    'threshold': 0.5,

    # --- Mixed precision ---
    'use_amp': True,

    # --- ImageNet normalization (DINOv2 was pretrained with these) ---
    'imagenet_mean': [0.485, 0.456, 0.406],
    'imagenet_std':  [0.229, 0.224, 0.225],

    # --- HuggingFace streaming dataset ---
    'hf_dataset': 'BIFOLD-BigEarthNetv2-0/BigEarthNet.txt',  # primary
    'hf_fallback': 'GFM-Bench/BigEarthNet',                 # if primary fails
    'val_fraction': 0.05,  # 5% of stream held out for validation
    'val_steps_per_epoch': 300,  # cap val batches per epoch for speed
}

# Official BigEarthNet-S2 43-class nomenclature (torchgeo class_sets[43]).
# MUST mirror backend/app/config.py CLASSES_43.
CLASSES_43 = [
    'Continuous urban fabric', 'Discontinuous urban fabric',
    'Industrial or commercial units', 'Road and rail networks and associated land',
    'Port areas', 'Airports', 'Mineral extraction sites', 'Dump sites',
    'Construction sites', 'Green urban areas', 'Sport and leisure facilities',
    'Non-irrigated arable land', 'Permanently irrigated land', 'Rice fields',
    'Vineyards', 'Fruit trees and berry plantations', 'Olive groves', 'Pastures',
    'Annual crops associated with permanent crops', 'Complex cultivation patterns',
    'Land principally occupied by agriculture, with significant areas of natural vegetation',
    'Agro-forestry areas', 'Broad-leaved forest', 'Coniferous forest', 'Mixed forest',
    'Natural grassland', 'Moors and heathland', 'Sclerophyllous vegetation',
    'Transitional woodland/shrub', 'Beaches, dunes, sands', 'Bare rock',
    'Sparsely vegetated areas', 'Burnt areas', 'Inland marshes', 'Peatbogs',
    'Salt marshes', 'Salines', 'Intertidal flats', 'Water courses',
    'Water bodies', 'Coastal lagoons', 'Estuaries', 'Sea and ocean',
]
assert len(CLASSES_43) == 43, f'Expected 43 classes, got {len(CLASSES_43)}'
CFG['classes'] = CLASSES_43

# Output directories under /kaggle/working
WORK = Path('/kaggle/working')
CKPT_DIR = WORK / 'checkpoints'; CKPT_DIR.mkdir(parents=True, exist_ok=True)
CURVE_DIR = WORK / 'training_curves'; CURVE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = WORK / 'logs'; LOG_DIR.mkdir(parents=True, exist_ok=True)
print(f'Outputs -> {WORK}')
print(f'Batch size: {CFG["batch_size"]} (effective ~{CFG["batch_size"]*CFG["grad_accum_steps"]})')

## 3 · Dataset (HuggingFace streaming, RGB-only)

We stream BigEarthNet-S2 to avoid materializing the full 66 GB on Kaggle's disk. Each sample is normalized to a common schema:

- `image`: a multi-band Sentinel-2 array (we extract **B04/R, B03/G, B02/B** only — DINOv2 expects 3-channel RGB).
- `labels`: a multi-hot vector of shape `(43,)`.

Different HF repos use slightly different field names, so `normalize_sample()` adapts per source.

In [ ]:
from datasets import load_dataset
import io

def normalize_sample(sample):
    """Convert an HF dataset row into (rgb_PIL, multi_hot_43_tensor).

    Handles field-name variation across BigEarthNet HF repos. RGB bands
    (B04=R, B03=G, B02=B) are selected from the S2 image. If the image is
    already 3-channel RGB (some repos store JPEG previews), it is used as-is.
    """
    # --- Image ---
    img = sample.get('image') or sample.get('img') or sample.get('s2')
    if img is None:
        raise KeyError(f'No image field in sample keys: {list(sample.keys())}')
    if not isinstance(img, Image.Image):
        img = Image.open(io.BytesIO(img['bytes'])) if isinstance(img, dict) else Image.open(io.BytesIO(img))

    # If multi-band numpy/Tensor, pull RGB = bands (B04, B03, B02).
    arr = np.asarray(img)
    if arr.ndim == 3 and arr.shape[2] >= 12:
        # torchgeo order: (B01,B02,B03,B04,...) -> RGB = indices [3,2,1]
        rgb = arr[:, :, [3, 2, 1]]
        img = Image.fromarray(np.uint8(np.clip(rgb, 0, 255)))
    elif arr.ndim == 2:
        img = img.convert('RGB')
    else:
        img = img.convert('RGB')

    # --- Labels -> multi-hot 43 ---
    labels = sample.get('labels') or sample.get('label') or sample.get('label_multi_hot')
    if labels is None:
        raise KeyError(f'No label field in sample keys: {list(sample.keys())}')

    if isinstance(labels, list) and len(labels) == 43:
        multi_hot = torch.tensor(labels, dtype=torch.float32)
    elif isinstance(labels, (list, tuple)) and all(isinstance(x, int) for x in labels) and max(labels) < 43:
        multi_hot = torch.zeros(43, dtype=torch.float32)
        multi_hot[list(labels)] = 1.0
    else:
        # Some repos store a ClassLabel sequence; coerce best-effort.
        multi_hot = torch.tensor(labels, dtype=torch.float32).reshape(-1)[:43]
        pad = 43 - multi_hot.shape[0]
        if pad > 0:
            multi_hot = F.pad(multi_hot, (0, pad))

    assert multi_hot.shape[0] == 43, f'Bad label shape {multi_hot.shape}'
    return img, multi_hot


def build_streaming_datasets():
    """Return (train_stream, val_stream) as HF IterableDatasets."""
    for repo in [CFG['hf_dataset'], CFG['hf_fallback']]:
        try:
            print(f'Attempting HF streaming from: {repo}')
            ds = load_dataset(repo, split='train', streaming=True)
            # Probe one sample to verify schema compatibility.
            probe = next(iter(ds))
            normalize_sample(probe)
            print(f'  OK — streaming from {repo}')
            break
        except Exception as e:
            print(f'  FAILED ({type(e).__name__}: {e}). Trying fallback.')
    else:
        raise RuntimeError(
            'Could not stream BigEarthNet-S2 from any HuggingFace repo. '
            'Options: (1) attach a BigEarthNet Kaggle Dataset and adapt the loader, '
            '(2) check HF token / internet setting.'
        )

    # Split the stream: validation is a fixed slice via skip/take.
    # NOTE: streaming split ratios are approximate; for a deterministic holdout
    # use the repo's official val split if present.
    val_stream = ds.take(20000)   # first 20k for validation
    train_stream = ds.skip(20000) # rest for training
    return train_stream, val_stream

train_stream, val_stream = build_streaming_datasets()

In [ ]:
# Wrap the streaming iterables in a torch Dataset + transforms.

train_tf = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=(0, 90)),         # random 90-deg rotation
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.RandomAffine(degrees=0, translate=0.1, scale=(0.9, 1.1)),
    T.Resize((CFG['image_size'], CFG['image_size'])),
    T.ToTensor(),
    T.Normalize(mean=CFG['imagenet_mean'], std=CFG['imagenet_std']),
])

val_tf = T.Compose([
    T.Resize((CFG['image_size'], CFG['image_size'])),
    T.ToTensor(),
    T.Normalize(mean=CFG['imagenet_mean'], std=CFG['imagenet_std']),
])

class StreamingBigEarthNet(Dataset):
    """Buffers a chunk of the HF stream into a torch Dataset.

    HF IterableDatasets can't be passed to torch DataLoader with num_workers>0
    cleanly, so we pre-fetch a buffer and refill when exhausted. This keeps
    memory bounded while still giving us shuffled, augmented batches.
    """
    def __init__(self, hf_stream, transform, buffer_size=2048, is_train=True):
        self.stream_iter = iter(hf_stream)
        self.transform = transform
        self.buffer_size = buffer_size
        self.is_train = is_train
        self._buffer = []
        self._refill()

    def _refill(self):
        self._buffer = []
        for _ in range(self.buffer_size):
            try:
                self._buffer.append(normalize_sample(next(self.stream_iter)))
            except StopIteration:
                break
        if self.is_train:
            random.shuffle(self._buffer)

    def __len__(self):
        return self.buffer_size

    def __getitem__(self, idx):
        if idx >= len(self._buffer):
            self._refill()
            if idx >= len(self._buffer):
                idx = 0
        img, label = self._buffer[idx]
        return self.transform(img), label

# Steps-per-epoch driven by buffer size + batch size.
BUFFER = CFG['batch_size'] * 50  # 50 refills worth per epoch
train_ds = StreamingBigEarthNet(train_stream, train_tf, buffer_size=BUFFER, is_train=True)
val_ds   = StreamingBigEarthNet(val_stream,   val_tf,   buffer_size=CFG['batch_size']*8, is_train=False)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)

train_steps = len(train_ds) // CFG['batch_size']
print(f'Train steps/epoch: {train_steps}  |  Val batches capped at: {CFG["val_steps_per_epoch"]}')

# Quick sanity check: one batch.
x, y = next(iter(train_loader))
print(f'Batch image: {tuple(x.shape)} {x.dtype} | labels: {tuple(y.shape)} {y.dtype}')
print(f'Label mean per class (should be sparse): {y.mean(0)[:5].tolist()} ...')

## 4 · Model

DINOv2-B backbone (`num_classes=0` → 768-d features) + custom head `Linear(768→512) → ReLU → Dropout(0.3) → Linear(512→43)`. **No sigmoid in forward** — logits only (BCEWithLogitsLoss applies sigmoid internally).

In [ ]:
from collections import OrderedDict

class MultiLabelDinoV2(nn.Module):
    def __init__(self, backbone=CFG['backbone'], dim=CFG['backbone_dim'],
                 num_classes=CFG['num_classes'], dropout=CFG['head_dropout'], pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0)
        # Head MUST match backend/app/models/dinov2.py exactly (checkpoint compat).
        self.classifier = nn.Sequential(OrderedDict([
            ('fc1',  nn.Linear(dim, 512)),
            ('act',  nn.ReLU(inplace=True)),
            ('drop', nn.Dropout(dropout)),
            ('fc2',  nn.Linear(512, num_classes)),
        ]))

    def forward(self, x):
        return self.classifier(self.backbone(x))  # raw logits (B, 43)

    # --- Phase freezing helpers (operate on timm ViT blocks) ---
    def freeze_all(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_last_n_blocks(self, n=2):
        # timm ViT exposes self.backbone.blocks as nn.Sequential of transformer blocks.
        blocks = getattr(self.backbone, 'blocks', None)
        if blocks is None:
            return
        total = len(blocks)
        for i in range(total - n, total):
            for p in blocks[i].parameters():
                p.requires_grad = True

    def unfreeze_full(self):
        for p in self.backbone.parameters():
            p.requires_grad = True

model = MultiLabelDinoV2(pretrained=True).to(DEVICE)

total = sum(p.numel() for p in model.parameters())
print(f'Model built. Total params: {total:,} (~{total/1e6:.0f}M)')

# Wrap in DataParallel if multiple GPUs (T4 x2).
if N_GPUS > 1:
    model = nn.DataParallel(model)
    print(f'Wrapped in DataParallel across {N_GPUS} GPUs.')

criterion = nn.BCEWithLogitsLoss()

## 5 · Training utilities

Metric computation, early stopping, checkpointing, and curve plotting.

In [ ]:
@torch.no_grad()
def compute_metrics(logits, targets, threshold=CFG['threshold']):
    """Return (f1_micro, f1_macro, hamming) for a batch."""
    probs = torch.sigmoid(logits).cpu().numpy()
    preds = (probs >= threshold).astype(int)
    tgt = targets.cpu().numpy().astype(int)
    f1m = f1_score(tgt, preds, average='micro', zero_division=0)
    f1M = f1_score(tgt, preds, average='macro', zero_division=0)
    h = hamming_loss(tgt, preds)
    return f1m, f1M, h


class EarlyStopping:
    def __init__(self, patience=CFG['es_patience'], min_delta=CFG['es_min_delta'], mode='max'):
        self.patience = patience; self.min_delta = min_delta
        self.mode = mode; self.best = -math.inf if mode == 'max' else math.inf
        self.counter = 0; self.triggered = False

    def step(self, value):
        improved = (value - self.best > self.min_delta) if self.mode == 'max' \
                   else (self.best - value > self.min_delta)
        if improved:
            self.best = value; self.counter = 0
            return False
        self.counter += 1
        if self.counter >= self.patience:
            self.triggered = True
            print(f'\n!!! Early stopping triggered (no {self.mode}-improvement for '
                  f'{self.patience} epochs, best={self.best:.4f}) !!!')
            return True
        return False


def save_checkpoint(model, epoch, optimizer, scheduler, train_metrics, val_metrics, is_best):
    """Save per-epoch + best checkpoints. Format matches backend loader."""
    raw = model.module if isinstance(model, nn.DataParallel) else model
    payload = {
        'epoch': epoch,
        'model_state': raw.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'train_metrics': train_metrics,
        'val_metrics': val_metrics,
        'config': CFG,
    }
    torch.save(payload, CKPT_DIR / f'model_epoch_{epoch}.pth')
    if is_best:
        torch.save(payload, CKPT_DIR / 'model_best.pth')
        print(f'   ★ new best (val_f1_micro={val_metrics["f1_micro"]:.4f}) -> model_best.pth')


def plot_curves(history, epoch):
    """2x2 grid: loss, F1, hamming loss, learning rate (log). Saved every 5 epochs."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    ep = range(1, len(history['train_loss']) + 1)

    axes[0,0].plot(ep, history['train_loss'], label='train')
    axes[0,0].plot(ep, history['val_loss'], label='val')
    axes[0,0].set_title('Loss'); axes[0,0].set_xlabel('epoch'); axes[0,0].legend(); axes[0,0].grid(True)

    axes[0,1].plot(ep, history['val_f1_micro'], label='F1 micro')
    axes[0,1].plot(ep, history['val_f1_macro'], label='F1 macro')
    axes[0,1].set_title('F1 Score'); axes[0,1].set_xlabel('epoch'); axes[0,1].legend(); axes[0,1].grid(True)

    axes[1,0].plot(ep, history['val_hamming'])
    axes[1,0].set_title('Hamming Loss (lower=better)'); axes[1,0].set_xlabel('epoch'); axes[1,0].grid(True)

    axes[1,1].plot(ep, history['lr'])
    axes[1,1].set_yscale('log')
    axes[1,1].set_title('Learning Rate'); axes[1,1].set_xlabel('epoch'); axes[1,1].grid(True)

    plt.tight_layout()
    out = CURVE_DIR / f'curves_epoch_{epoch}.png'
    plt.savefig(out, dpi=120); plt.close()
    print(f'   Saved curves -> {out}')

print('Utilities ready.')

## 6 · Three-phase training loop

- **Phase 1** (5 ep): head only, lr 1e-3
- **Phase 2** (10 ep): + last 2 transformer blocks, lr 1e-4
- **Phase 3** (≤15 ep): full fine-tune, lr 1e-5

Each phase rebuilds the optimizer + ReduceLROnPlateau. FP16 autocast + GradScaler throughout. Checkpoint saved every epoch; best saved on val F1-micro improvement; curves every 5 epochs.

In [ ]:
writer = SummaryWriter(log_dir=str(LOG_DIR / 'tensorboard'))
history = {'train_loss': [], 'val_loss': [], 'val_f1_micro': [],
           'val_f1_macro': [], 'val_hamming': [], 'lr': [], 'epoch_time': []}
early = EarlyStopping()
best_f1_micro = -1.0
global_epoch = 0

def unwrap(m): return m.module if isinstance(m, nn.DataParallel) else m

for phase in CFG['phases']:
    pname, n_epochs, lr, unfreeze = phase['name'], phase['epochs'], phase['lr'], phase['unfreeze']
    print(f'\n{"="*70}\nPHASE: {pname}  |  epochs={n_epochs}  |  lr={lr}  |  unfreeze={unfreeze}\n{"="*70}')

    # --- Freeze policy ---
    raw = unwrap(model)
    raw.freeze_all()
    if unfreeze == 'last_2_blocks':
        raw.unfreeze_last_n_blocks(2)
    elif unfreeze == 'full':
        raw.unfreeze_full()
    # head is always trainable
    for p in raw.classifier.parameters():
        p.requires_grad = True

    trainable = [p for p in model.parameters() if p.requires_grad]
    print(f'  Trainable params: {sum(p.numel() for p in trainable):,}')

    optimizer = torch.optim.AdamW(trainable, lr=lr, betas=CFG['betas'], weight_decay=CFG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode=CFG['sched_mode'], factor=CFG['sched_factor'],
        patience=CFG['sched_patience'], min_lr=CFG['sched_min_lr'])
    scaler = GradScaler(enabled=CFG['use_amp'])

    for epoch in range(1, n_epochs + 1):
        global_epoch += 1
        t0 = time.time()
        model.train()

        # --- Train ---
        running_loss, optimizer.zero_grad(set_to_none=True), accum = 0.0, None, 0
        optimizer.zero_grad(set_to_none=True)
        for step, (imgs, labels) in enumerate(train_loader, 1):
            imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            with autocast(enabled=CFG['use_amp']):
                logits = model(imgs)
                loss = criterion(logits, labels) / CFG['grad_accum_steps']
            scaler.scale(loss).backward()
            running_loss += loss.item() * CFG['grad_accum_steps']
            accum += 1
            if accum >= CFG['grad_accum_steps']:
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
                accum = 0
            if step % 50 == 0:
                print(f'  epoch {global_epoch} step {step}/{train_steps} loss={loss.item()*CFG["grad_accum_steps"]:.4f}', end='\r')
        train_loss = running_loss / max(1, train_steps)

        # --- Validate ---
        model.eval()
        v_loss, all_logits, all_tgt = 0.0, [], []
        with torch.no_grad():
            for vi, (imgs, labels) in enumerate(val_loader):
                if vi >= CFG['val_steps_per_epoch']: break
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                with autocast(enabled=CFG['use_amp']):
                    logits = model(imgs)
                    v_loss += criterion(logits, labels).item()
                all_logits.append(logits.float().cpu())
                all_tgt.append(labels.cpu())
        val_loss = v_loss / max(1, min(CFG['val_steps_per_epoch'], len(val_loader)))
        logits_cat = torch.cat(all_logits); tgt_cat = torch.cat(all_tgt)
        f1m, f1M, h = compute_metrics(logits_cat, tgt_cat)

        cur_lr = optimizer.param_groups[0]['lr']
        dt = time.time() - t0

        # --- Record ---
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1_micro'].append(f1m)
        history['val_f1_macro'].append(f1M)
        history['val_hamming'].append(h)
        history['lr'].append(cur_lr)
        history['epoch_time'].append(dt)

        writer.add_scalar('Loss/train', train_loss, global_epoch)
        writer.add_scalar('Loss/val', val_loss, global_epoch)
        writer.add_scalar('F1/micro', f1m, global_epoch)
        writer.add_scalar('F1/macro', f1M, global_epoch)
        writer.add_scalar('Hamming/val', h, global_epoch)
        writer.add_scalar('LR', cur_lr, global_epoch)

        print(f'\n  epoch {global_epoch} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} '
              f'f1_micro={f1m:.4f} f1_macro={f1M:.4f} hamming={h:.4f} lr={cur_lr:.2e} time={dt:.1f}s')

        # --- Scheduler + checkpoint ---
        scheduler.step(f1m)
        is_best = f1m > best_f1_micro + CFG['es_min_delta']
        if is_best: best_f1_micro = f1m
        save_checkpoint(model, global_epoch, optimizer, scheduler,
                        {'loss': train_loss},
                        {'loss': val_loss, 'f1_micro': f1m, 'f1_macro': f1M, 'hamming': h},
                        is_best)

        if global_epoch % 5 == 0:
            plot_curves(history, global_epoch)

        # --- Early stopping ---
        if early.step(f1m):
            break
    if early.triggered:
        break

writer.close()
print(f'\nTraining finished. Best val F1 micro: {best_f1_micro:.4f}')

## 7 · Export training history + final curves

In [ ]:
# Final history JSON
with open(LOG_DIR / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print(f'Saved -> {LOG_DIR / "training_history.json"}')

# Final curves plot
plot_curves(history, global_epoch)

print('\n--- Checkpoints ---')
for p in sorted(CKPT_DIR.glob('*.pth')):
    print(f'  {p.name}  ({p.stat().st_size/1e6:.1f} MB)')
print('\nRemember: download model_best.pth from the Output tab and place it in backend/checkpoints/')